# Python como cliente de un grafo: Neo4j

Neo4j es una base de grafos: almacena nodos, propiedades y relaciones. Es una forma
de trabajar con datos NoSQL; otras bases NoSQL usan documentos o pares clave–valor.
NoSQL no significa ausencia de consultas: Neo4j utiliza **Cypher**, no SQL.

Retomaremos los documentos del catálogo y sus autores. Una relación `WROTE`
permite consultar qué autor escribió documentos sobre un tema. La base funciona
en un servidor; el paquete `neo4j` es su cliente Python, no instala ese servidor.

Esta parte es una **demostración sin ejercicios**. Para ejecutarla, sigue la
[preparación de Desktop](./README.md#neo4j-desktop-opcional), activa `RUN_NEO4J`
y reinicia la notebook. Si no instalas Neo4j, puedes leer los ejemplos y sus salidas esperadas.


In [1]:
from getpass import getpass

from neo4j import GraphDatabase

from sample_data import AUTHORS, DOCUMENTS

RUN_NEO4J = False
NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_DATABASE = "neo4j"
password = getpass("Neo4j password: ") if RUN_NEO4J else ""


## 1. Conectar y cerrar el driver

El driver administra las conexiones al servidor. `verify_connectivity()` comprueba
que puede conectarse; crear el driver por sí solo no verifica la conexión.
`auth` recibe usuario y contraseña. `getpass()` pide la contraseña sin mostrarla.
El bloque `with` cierra el driver al terminar.


In [2]:
if RUN_NEO4J:
    with GraphDatabase.driver(NEO4J_URI, auth=("neo4j", password)) as driver:
        driver.verify_connectivity()
        print("Connection established")
else:
    print("Neo4j demonstration disabled; enable it after starting your local database.")


Neo4j demonstration disabled; enable it after starting your local database.


## 2. Crear el ejemplo

Usamos etiquetas `LessonAuthor` y `LessonDocument` para identificar nuestros nodos.
`MERGE` busca el nodo o la relación y lo crea si falta; `SET` actualiza propiedades.
Los nombres `$document_id`, `$title`, etc. son parámetros de Cypher, separados de
los valores de Python. El patrón `(author)-[:WROTE]->(document)` representa la relación.

`driver.session(database=...)` abre una sesión para ejecutar consultas. Su `with`
la cierra antes que el driver. `consume()` completa la consulta antes de continuar;
estas escrituras simples se confirman como transacciones automáticas.


In [3]:
create_query = """
    MERGE (author:LessonAuthor {name: $author})
    MERGE (document:LessonDocument {id: $document_id})
    SET document.title = $title, document.topic = $topic, document.pages = $pages
    MERGE (author)-[:WROTE]->(document)
"""
if RUN_NEO4J:
    with GraphDatabase.driver(NEO4J_URI, auth=("neo4j", password)) as driver:
        with driver.session(database=NEO4J_DATABASE) as session:
            for document_id, title, topic, pages in DOCUMENTS:
                session.run(
                    create_query,
                    document_id=document_id,
                    title=title,
                    topic=topic,
                    pages=pages,
                    author=AUTHORS[document_id],
                ).consume()
    print("Six documents and their authors are ready")


## 3. Consultar relaciones desde Python

`MATCH` describe el patrón que buscamos, `WHERE` filtra y `RETURN` elige campos.
Aquí recuperamos al autor y los documentos sobre `retrieval`. `result.data()`
materializa una lista de diccionarios **dentro de la sesión**, antes de cerrarla.
Para este ejemplo pequeño es suficiente; no estamos cargando un grafo completo.


In [4]:
read_query = """
    MATCH (author:LessonAuthor)-[:WROTE]->(document:LessonDocument)
    WHERE document.topic = $topic
    RETURN author.name AS author, document.title AS title
    ORDER BY document.id
"""
if RUN_NEO4J:
    with GraphDatabase.driver(NEO4J_URI, auth=("neo4j", password)) as driver:
        with driver.session(database=NEO4J_DATABASE) as session:
            records = session.run(read_query, topic="retrieval").data()
    for record in records:
        print(record["author"], "->", record["title"])


Salida esperada con los datos proporcionados:

```text
Sam -> Text embeddings
Sam -> Graph search
```

En SQLite consultamos filas; en Neo4j consultamos un patrón de relaciones. En ambos
casos Python envía parámetros y transforma resultados en estructuras conocidas.
Una base de grafos no sustituye siempre a una tabla: la elección depende de los
datos y las consultas que necesita la aplicación.

Fuentes: [conexión](https://neo4j.com/docs/python-manual/current/connect/),
[consultas y parámetros](https://neo4j.com/docs/python-manual/current/query-simple/),
[sesiones y resultados](https://neo4j.com/docs/python-manual/current/transactions/).
